In [1]:
import pandas as pd
import os
import ast

from utils import log2pd, dot2bp, f1_score

En este Notebook lo que se busca es armar las tablas para el caso famfold, que fue entrenado 15 / 50 / 100/ 150 / 200 epocas.

Quiero obtener tablas con columnas: `familia, epoca, f1`.

# Build the pipeline
Checking with random test.log

In [2]:
DATA_REF_PATH = "/home/gkulemeyer/Documents/Repos/RNA-analysis/DataAnalysis/data/sources/ArchiveII.csv"
df_ref = pd.read_csv(DATA_REF_PATH, index_col="id")
df_ref["base_pairs"] = df_ref["base_pairs"].apply(ast.literal_eval)

df_ref.head(2)

,sequence,structure,base_pairs,len
id,,,,
5s_Acholeplasma-laidlawii-1,UCUGGUGACGAUAGGUAAGAUGGUUCACCUGUUCCCAUCCCGAACA...,((((((((......((((((((....((((((.............)...,"[[1, 111], [2, 110], [3, 109], [4, 108], [5, 1...",112
5s_Acidovorax-temperans-1,UGCCUGAUGACCAUAGCAAGUUGGUACCACUCCUUCCCAUCCCGAA...,.(((((((((.....((((((((.....((((((...............,"[[2, 115], [3, 114], [4, 113], [5, 112], [6, 1...",115


In [3]:
########################################################################
DATA_ROOT = "../BRONZE/ArchiveII_fam-fold/" ########### DATA ###########
########################################################################
fams = os.listdir(DATA_ROOT)
fams.sort()
print(fams)
eps = os.listdir(DATA_ROOT + fams[0])
eps.sort()
print(eps)

['16s', '23s', '5s', 'RNaseP', 'grp1', 'srp', 'tRNA', 'telomerase', 'tmRNA']
['ep_015', 'ep_050', 'ep_100', 'ep_150', 'ep_200']


In [4]:
# for f in fams: for ep in eps: ...
df = log2pd(DATA_ROOT + fams[0] + "/" + eps[0] + "/test.log")
print(df.shape)
df.head(2)

(66, 3)


,id,sequence,structure
0,16s_A.pyrophilus_domain4,CCGCCCGUCACGCCACGGAAGUCGGUCCGGCCGGAAGUCCCCGAGC...,((.(..........((....)).(...)........((.(((.......
1,16s_C.psittaci_domain3,AAAGAAUUGACGGGGGCCCGCACAAGCAGUGGAGCAUGUGGUUUAA...,.(.....(((((((((((..(((.((((((...(.(.)....)).....


In [5]:
df["base_pairs"] = df["structure"].apply(dot2bp)

df["test_f1"] = df.apply(
    lambda row: f1_score(
        df_ref.loc[row.id, "base_pairs"],  # lista de pares ref
        row["base_pairs"],  # lista de pares pred
    ),
    axis=1,
)
df.test_f1.mean()

np.float64(0.3092734513536199)

# Make table

In [6]:
def extract_info(path, fam, source, ep):
    # for f in fams: for ep in eps: ...
    EPOCHS = {"ep_015": 15, "ep_050": 50, "ep_100": 100, "ep_150": 150, "ep_200": 200}
    df = log2pd(path)
    df["base_pairs"] = df["structure"].apply(dot2bp)

    df["test_f1"] = df.apply(
        lambda row: f1_score(
            df_ref.loc[row.id, "base_pairs"],  # lista de pares ref
            row["base_pairs"],  # lista de pares pred
        ),
        axis=1,
    )
    return {
        "fam": fam,
        "source": source,
        "epoch": EPOCHS[ep],
        "test_f1": df.test_f1.mean(),
    }

In [7]:
source = "famfold"
rows = []
for f in fams:
    for ep in eps:
        test_log = DATA_ROOT + f + "/" + ep + "/test.log"
        row = extract_info(test_log, f, source, ep)
        rows.append(row)
table = pd.DataFrame(rows)
table.sample(5)

,fam,source,epoch,test_f1
9,23s,famfold,200,0.365124
29,srp,famfold,200,0.195454
2,16s,famfold,100,0.366523
25,srp,famfold,15,0.181983
14,5s,famfold,200,0.484213


In [8]:
SAVE_PATH = "../SILVER/ArchiveII_fam-fold/f1_by_epoch/"
os.makedirs(SAVE_PATH, exist_ok=True)
table.to_csv(SAVE_PATH + "test_f1.csv", index=False)

In [10]:
!ls  ../BRONZE/

ArchiveII_fam-fold  ArchiveII_rnadist_100  ArchiveII_samples_200
ArchiveII_hc_100    ArchiveII_rnadist_200  ArchiveII_samples_400
ArchiveII_hc_200    ArchiveII_rnadist_400
ArchiveII_hc_400    ArchiveII_samples_100
